# Dual-Pipeline PDF Ingestion (MinerU visuals + Qwen-VL page text)

Replaces MinerU's own text/OCR extraction (quality issues on this
document set) with Qwen2.5-VL full-page transcription, while keeping
MinerU for what it's good at: layout detection and cropping out
figures, charts, diagrams, and tables as images.

```
PDF
 |- Thread A: MinerU (images/tables/diagrams only) --------.
 `- Thread B: PyMuPDF render -> Qwen-VL (text only) -------|
                                                            v
                                                      merge by page
```

Both threads run as separable sequential stages (not literal concurrent
threads) to avoid MPS device contention between MinerU's own OCR/layout
models and Qwen2.5-VL.

**No captioning at this stage** — Thread A only extracts and saves
image crops (with page_idx metadata); captioning is a later stage.

**Merge semantics:** the two streams share no common item-level ID
(Thread B never sees MinerU's content_list), so they're joined on
`page_idx` alone into a page-grouped JSON: each page maps to its
transcribed text plus a list of that page's visual items (type,
sub_type, img_path). See `ingestion.merge_by_page` for details.

**Isolated env note:** MinerU runs via the separate `.venv-mineru/` venv
(see `pdf_parsing_mineru.ipynb`) — its `transformers<5.0.0` pin conflicts
with this project's `transformers>=5.14.1` (needed by Qwen2.5-VL).

## Setup

In [1]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent  # notebook lives in notebooks/
sys.path.insert(0, str(PROJECT_ROOT / "src"))

MINERU_BIN = PROJECT_ROOT / ".venv-mineru" / "bin" / "mineru"
assert MINERU_BIN.exists(), (
    f"MinerU venv not found at {MINERU_BIN}. Create it with:\n"
    f"  uv venv .venv-mineru --python 3.13\n"
    f'  uv pip install --python .venv-mineru "mineru[core]"'
)

PDF_PATH = PROJECT_ROOT / "notebooks" / "Anatomy of Neck - Basic of DEMN.pdf_origin.pdf"
OUTPUT_ROOT = PROJECT_ROOT / "output"
assert PDF_PATH.exists(), PDF_PATH

PDF_STEM = PDF_PATH.stem  # MinerU strips the .pdf extension for its run dir name
RUN_DIR = OUTPUT_ROOT / PDF_STEM / "auto"  # "auto" = pipeline backend's dir suffix

PROJECT_ROOT, PDF_PATH

(PosixPath('/Users/michaeleko/Documents/Works/aiml-institute/challenge-2/intelligent-tutoring-system-for-medical-student'),
 PosixPath('/Users/michaeleko/Documents/Works/aiml-institute/challenge-2/intelligent-tutoring-system-for-medical-student/notebooks/Anatomy of Neck - Basic of DEMN.pdf_origin.pdf'))

## Thread A: MinerU — visuals only

Runs MinerU (`pipeline` backend — `hybrid-engine` crashes on MPS, see
`pdf_parsing_mineru.ipynb`). We only keep its image/chart/table/diagram
crops; all text/paragraph/title fields from `content_list_v2` are
ignored in favor of Thread B.

In [2]:
import subprocess

cmd = [
    str(MINERU_BIN),
    "-p", str(PDF_PATH),
    "-o", str(OUTPUT_ROOT),
    "-b", "pipeline",
    "-m", "auto",
]

result = subprocess.run(cmd, cwd=PROJECT_ROOT, capture_output=True, text=True)
print(result.stdout[-4000:])
if result.returncode != 0:
    print(result.stderr[-4000:])
result.returncode

Start MinerU FastAPI Service: http://127.0.0.1:51079
API documentation: http://127.0.0.1:51079/docs



0

In [3]:
from ingestion.clean_content_list import build_clean_content_list

CONTENT_LIST_V2_PATH = RUN_DIR / f"{PDF_STEM}_content_list_v2.json"
CONTENT_LIST_PATH = build_clean_content_list(CONTENT_LIST_V2_PATH)

with open(CONTENT_LIST_PATH) as f:
    content_list = json.load(f)

visual_items = [
    item for item in content_list
    if item.get("type") in {"image", "chart", "table", "diagram"} and item.get("img_path")
]
len(visual_items), visual_items[0] if visual_items else None

(86,
 {'type': 'image',
  'img_path': 'images/06c56221cfb037a49acd3b665c2157e9f55de83cdb1f33130bad7a618708a59d.jpg',
  'image_caption': [],
  'image_footnote': [],
  'content': '',
  'sub_type': None,
  'bbox': [547, 54, 997, 997],
  'page_idx': 0})

In [4]:
from ingestion.merge_by_page import visuals_by_page

visuals_page_map = visuals_by_page(content_list, PDF_STEM)

sum(len(v) for v in visuals_page_map.values())

86

## Thread B: PyMuPDF render + Qwen-VL — page text only

Renders every page to a PNG, then transcribes body text with
Qwen2.5-VL.

In [5]:
from ingestion.page_render import render_pages

PAGE_IMAGES_DIR = RUN_DIR / "rendered_pages"
page_image_paths = render_pages(PDF_PATH, PAGE_IMAGES_DIR)

len(page_image_paths), page_image_paths[0]

(73,
 PosixPath('/Users/michaeleko/Documents/Works/aiml-institute/challenge-2/intelligent-tutoring-system-for-medical-student/output/Anatomy of Neck - Basic of DEMN.pdf_origin/auto/rendered_pages/page_0000.png'))

In [ ]:
from tqdm.auto import tqdm

from captioning.qwen_vl import QwenVLCaptioner

captioner = QwenVLCaptioner("Qwen/Qwen3-VL-8B-Instruct")

page_texts: dict[int, str] = {}
for page_idx, image_path in enumerate(tqdm(page_image_paths, desc="Transcribing pages")):
    page_texts[page_idx] = captioner.transcribe_page(str(image_path), max_new_tokens=512)

len(page_texts)

/Users/michaeleko/Documents/Works/aiml-institute/challenge-2/intelligent-tutoring-system-for-medical-student/.venv-mineru/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Transcribing pages:   0%|          | 0/73 [00:00<?, ?it/s]The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 5/5 [00:00<00:00, 87.24it/s]
The following generation flags are not valid and may be ignored: ['temper

73

## Classify visual relevance (semantic vs. decorative)

For each visual on a page, ask Qwen-VL whether it supports the page's
transcribed text or is purely decorative. Reuses the still-loaded
Thread B captioner (`classify_image_relevance`), so no extra model
load/unload cycle — released after this step.

In [7]:
for page_idx, visuals in tqdm(visuals_page_map.items(), desc="Classifying visuals"):
    slide_text = page_texts.get(page_idx, "")
    for visual in visuals:
        img_path = RUN_DIR / visual.img_path
        visual.relevance = captioner.classify_image_relevance(str(img_path), slide_text)

captioner.unload()
sum(v.relevance == "decorative" for visuals in visuals_page_map.values() for v in visuals)

Classifying visuals: 100%|██████████| 70/70 [03:51<00:00,  3.31s/it]


7

## Merge by page

In [8]:
from ingestion.merge_by_page import render_merged_document

merged_document = render_merged_document(page_texts, visuals_page_map)

len(merged_document), merged_document[str(0)] if "0" in merged_document else None

(73,
 {'text': '# Anatomy of the Neck\n## Basic of DEMN System',
  'visuals': [{'item_id': 'Anatomy of Neck - Basic of DEMN.pdf_origin#p0#2',
    'type': 'image',
    'sub_type': None,
    'img_path': 'images/06c56221cfb037a49acd3b665c2157e9f55de83cdb1f33130bad7a618708a59d.jpg',
    'relevance': 'semantic'}]})

In [9]:
merged_path = RUN_DIR / f"{PDF_STEM}_dual_pipeline_merged.json"
with open(merged_path, "w") as f:
    json.dump(merged_document, f, indent=2)

merged_path

PosixPath('/Users/michaeleko/Documents/Works/aiml-institute/challenge-2/intelligent-tutoring-system-for-medical-student/output/Anatomy of Neck - Basic of DEMN.pdf_origin/auto/Anatomy of Neck - Basic of DEMN.pdf_origin_dual_pipeline_merged.json')